# F5 · Madde 6 — Host düzeyindeki güven mekanizması seyrek saldırılarda ne yapıyor?
**Profesörün notu:** *Host düzeyindeki güven mekanizmasını seyrek saldırılar açısından da değerlendir.*

Host sayacı, bir hostun W akışlık penceresinde p ≤ α+β olan akışları sayar ve sayı k\*'a ulaşınca hostu yükseltir.
Az sayıda kötü akışı çok sayıda normal akışın arasına yayan bir saldırgan k\*'ın altında kalabilir. Bu defter mekanizmayı
**değiştirmez, ölçer**:

* pencere başına saldırı payı ρ ∈ {%1, %2, %5, %10, %25, %50, %100}; pencerede round(ρ·W) saldırı akışı bulunur
  (hiç saldırı akışı içermeyen birleşimler atlanır ve tabloda yer almaz);
* W ∈ {20, 50, 100, 200, 500} (kalibrasyon için yeterli normal akışı olan host varsa);
* aynı operasyonel bütçe (cihaz başına haftada ≤ 1 yanlış yükseltme), k\* yine normal-doğrulama pencerelerinden;
* karşılaştırma: yalnız ANOMALY akışlarını sayan sayaç (kendi k\* değeriyle);
* tohumlar 0–9 (02 defterinde eğitilen modeller), 9 saldırı ailesinin tümü.

**Dürüstlük notları:** Sürüm B'de zaman damgası yoktur; pencereler rastgele çekilir (**simülasyon**). "Günlük tespit"
sütunu pencerelerin bağımsız olduğu varsayımıyla hesaplanan bir yaklaşımdır. Bu analiz önceden kaydedilmedi; **keşif
amaçlıdır**. ρ → 0 iken tespit oranı yanlış yükseltme oranına yaklaşır; tablolar bu yüzden ikisini birlikte verir.

In [ ]:
# ---- Kurulum (her defterde aynı): Drive'ı bağla, ortak kodu içe aktar ----
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
PROJECT_DIR = os.environ.get("TRUSTIDS_PROJECT", "/content/drive/MyDrive/TrustFed-TinyIDS-Final")
QUICK = os.environ.get("TRUSTIDS_QUICK", "0") == "1"   # hızlı mod YALNIZCA kodu test etmek içindir
# Devam modu: Colab oturumu kapanırsa, Drive'a kaydedilmiş tohumlar/adımlar atlanır ve kalan yerden devam edilir.
# Her şeyi baştan hesaplamak için: os.environ["TRUSTIDS_RESUME"] = "0"
RESUME = os.environ.get("TRUSTIDS_RESUME", "1") == "1"
sys.path.insert(0, PROJECT_DIR)

from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from trustids.config import Config
from trustids import utils

# Veri kümesi seçimi: varsayılan TON_IoT; ikinci veri kümesi için TRUSTIDS_CONFIG=edge_iiotset.json
CONFIG_NAME = os.environ.get("TRUSTIDS_CONFIG", "default.json")
CFG_PATH = Path(PROJECT_DIR) / "configs" / CONFIG_NAME
if CONFIG_NAME != "default.json" and not CFG_PATH.exists():
    raise FileNotFoundError(f"{CFG_PATH} bulunamadı: configs/ klasörünü de Drive'a yükleyin.")
cfg = Config.load(CFG_PATH) if CFG_PATH.exists() else Config(project_dir=PROJECT_DIR)
cfg.project_dir = PROJECT_DIR
if QUICK:
    cfg.train_seeds, cfg.validity_resplits, cfg.epochs = cfg.train_seeds[:2], 30, 15
    cfg.random_control_repeats, cfg.bootstrap_repeats = 10, 200
    cfg.lofo_seeds, cfg.fl_orig_rounds = cfg.lofo_seeds[:2], 3
    cfg.resolution_cal_windows, cfg.resolution_eval_windows = 600, 600
cfg.check()
ART = cfg.artifacts
FIG = Path(PROJECT_DIR) / cfg.figures_dir; FIG.mkdir(exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("Proje klasörü:", PROJECT_DIR, "| hızlı mod:", QUICK, "| devam modu:", RESUME)
print("Drive'da kayıtlı ilerleme:", utils.progress(cfg))

In [ ]:
from trustids import data as D, conformal as C, evaluation as E, resolution as R, sparse as SPR
P = D.load_prepared(ART / "prepared")
SDIR = ART / "sparse"; SDIR.mkdir(exist_ok=True)
FP = utils.fingerprint(cfg, ["sparse_rho", "sparse_windows", "sparse_mixed_windows", "resolution_flows_per_day",
                             "resolution_budget_per_week", "resolution_cal_windows", "resolution_eval_windows", "alpha", "beta"])
RUN = utils.new_run_dir(Path(PROJECT_DIR) / cfg.runs_dir, "final_F5_seyrek_saldiri")
utils.save_json(utils.environment_info(), RUN / "environment.json"); cfg.save(RUN / "config_used.json")
HOSTMAP = ART / "resolution" / "host_rows.csv.gz"
if HOSTMAP.exists():
    ROWS = pd.read_csv(HOSTMAP, keep_default_na=False); ROWS["split"] = ROWS["split"].replace({"": None})
else:
    raw = D.load_raw(cfg.data_path, cfg); ROWS = R.map_rows(raw, P, cfg)
    HOSTMAP.parent.mkdir(exist_ok=True); ROWS.to_csv(HOSTMAP, index=False)
ROWS = ROWS[ROWS["split"].isin(["benign_val", "benign_test", "dev_test", "heldout_test"])].reset_index(drop=True)
SEEDS = E.available_seeds(cfg)
hv = ROWS[ROWS["split"] == "benign_val"].groupby("host").size()
print("tohumlar:", SEEDS, "| ρ:", cfg.sparse_rho, "| W:", cfg.sparse_windows)
print("normal-doğrulama: W akışa ulaşan host sayısı:", {W: int((hv >= W).sum()) for W in cfg.sparse_windows})

In [ ]:
for seed in SEEDS:
    f = SDIR / f"seed{seed}_{FP}.csv"
    if RESUME and f.exists():
        print("tohum", seed, "Drive'da kayıtlı, atlanıyor (devam modu)"); continue
    S = E.load_scores(cfg, seed)
    Sae = {sp: S[("ae", sp)] for sp in ["benign_cal", "benign_val", "benign_test", "dev_test", "heldout_test"]}
    pd.DataFrame(SPR.evaluate_seed(ROWS, Sae, cfg, seed, np.random.default_rng(50_000 + seed))).to_csv(f, index=False)
    print("tohum", seed, "tamam (Drive'a kaydedildi)", flush=True)
RES = pd.concat([pd.read_csv(SDIR / f"seed{s}_{FP}.csv") for s in SEEDS], ignore_index=True)
RES.to_csv(RUN / "F5_tum_tohumlar.csv", index=False)
SKIP = RES[RES["durum"] != "tamam"][["W", "durum"]].drop_duplicates()
MAIN = RES[RES["durum"] == "tamam"].copy()
print("atlanan pencere boyutları:", SKIP.to_dict(orient="records") if len(SKIP) else "yok")

## Normal trafikte maliyet (her W, F ve sayaç için; hedef haftada ≤ 1)

In [ ]:
COST = (MAIN.drop_duplicates(["tohum", "W", "F", "sayaç"]).groupby(["sayaç", "W", "F"])
            [["k_yildiz", "α_w", "normal_yanlis_yukseltme_pencere", "normal_haftalik_ort", "normal_haftalik_en_kotu_host"]].agg(["mean", "std"]).round(4))
COST.columns = [f"{a}_{b}" for a, b in COST.columns]; COST.to_csv(RUN / "F5_normal_maliyet.csv"); COST

## Tespit: saldırı payı ρ'ya göre pencere düzeyinde tespit oranı (önce tohum içinde aile ortalaması, sonra tohum ortalaması ± std)

In [ ]:
g1 = MAIN.groupby(["tohum", "sayaç", "W", "F", "grup", "ρ"], as_index=False)[["pencere_tespit", "gunluk_tespit_bagimsiz_pencere", "saldiri_akisi_pencerede"]].mean()
DET = g1.groupby(["sayaç", "W", "F", "grup", "ρ"])[["pencere_tespit", "gunluk_tespit_bagimsiz_pencere", "saldiri_akisi_pencerede"]].agg(["mean", "std"]).round(4)
DET.columns = [f"{a}_{b}" for a, b in DET.columns]; DET.to_csv(RUN / "F5_tespit_grup.csv")
FAMT = MAIN.groupby(["sayaç", "W", "F", "grup", "aile", "ρ"])[["pencere_tespit", "akis_ANOMALY", "akis_isaretli"]].mean().round(4)
FAMT.to_csv(RUN / "F5_tespit_aile.csv")
F0 = max(cfg.resolution_flows_per_day)
DET.xs(("ANOMALY+UNCERTAIN", F0), level=("sayaç", "F"))["pencere_tespit_mean"].unstack("ρ")

In [ ]:
DET.xs(("ANOMALY+UNCERTAIN", F0), level=("sayaç", "F"))["gunluk_tespit_bagimsiz_pencere_mean"].unstack("ρ")

In [ ]:
Ws = sorted(MAIN["W"].unique())
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, grp in zip(axs, ["görülmemiş", "bilinen"]):
    for W in Ws:
        for cname, ls in [("ANOMALY+UNCERTAIN", "-"), ("yalnız ANOMALY", ":")]:
            q = g1[(g1["sayaç"] == cname) & (g1["W"] == W) & (g1["F"] == F0) & (g1["grup"] == grp)].groupby("ρ")["pencere_tespit"].mean()
            if len(q): ax.plot(q.index, 100 * q.values, ls, marker="o", ms=3, lw=1.2, label=f"W={W} · {cname}")
    ax.set_xscale("log"); ax.set_xlabel("pencere başına saldırı payı ρ (log)"); ax.set_title(f"{grp} aileler · F = {F0}/gün")
axs[0].set_ylabel("pencere düzeyinde tespit (%)"); axs[1].legend(fontsize=6, ncol=2)
fig.suptitle("Seyrek saldırılarda host sayacı (simülasyon; tohum ortalaması)"); fig.tight_layout(); fig.savefig(RUN / "F5_seyrek_saldiri.png", dpi=150)
print("Sonuçlar:", RUN)

In [ ]:
# Colab: Drive'a yazılan tüm dosyaların kaydedildiğinden emin ol (oturumu kapatmadan önce mutlaka çalıştırın)
print("Kaydedilen ilerleme:", utils.progress(cfg))
if IN_COLAB:
    drive.flush_and_unmount()
    print("Drive kaydedildi ve ayrıldı. Sonraki defter Drive'ı yeniden bağlar.")